## Similarity Exercise

In [ ]:
# !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer
import faiss

In this exercise, you've been provided the title and abstract of 500 recent machine learning research papers posted on arXiv.org.

In [ ]:
articles = pd.read_csv('arxiv_papers.csv')
articles.head()

,title,abstract,url
0,GoT-R1: Unleashing Reasoning Capability of MLL...,Visual generation models have made remarkable ...,http://arxiv.org/abs/2505.17022v1
1,Delving into RL for Image Generation with CoT:...,Recent advancements underscore the significant...,http://arxiv.org/abs/2505.17017v1
2,Interactive Post-Training for Vision-Language-...,"We introduce RIPT-VLA, a simple and scalable r...",http://arxiv.org/abs/2505.17016v1
3,When Are Concepts Erased From Diffusion Models?,"Concept erasure, the ability to selectively pr...",http://arxiv.org/abs/2505.17013v1
4,Understanding Prompt Tuning and In-Context Lea...,Prompting is one of the main ways to adapt a p...,http://arxiv.org/abs/2505.17010v1


In [ ]:
i = 0
print(f'Title: {articles.loc[i,"title"]}\n')
print(f'Text: {articles.loc[i,"abstract"]}')

Title: GoT-R1: Unleashing Reasoning Capability of MLLM for Visual Generation with Reinforcement Learning

Text: Visual generation models have made remarkable progress in creating realistic
images from text prompts, yet struggle with complex prompts that specify
multiple objects with precise spatial relationships and attributes. Effective
handling of such prompts requires explicit reasoning about the semantic content
and spatial layout. We present GoT-R1, a framework that applies reinforcement
learning to enhance semantic-spatial reasoning in visual generation. Building
upon the Generation Chain-of-Thought approach, GoT-R1 enables models to
autonomously discover effective reasoning strategies beyond predefined
templates through carefully designed reinforcement learning. To achieve this,
we propose a dual-stage multi-dimensional reward framework that leverages MLLMs
to evaluate both the reasoning process and final output, enabling effective
supervision across the entire generation pipeli

Let's try out a variety of ways of vectorizing and searching for semantically-similar papers.

### Method 1: Bag of Words

Fit a CountVectorizer to the abstracts of the articles with all of the defaults.  Then vectorize the dataset using the fit vectorizer.

In [ ]:
vect = CountVectorizer().fit(articles['abstract'])
vect_articles = vect.transform(articles['abstract'])

**Question:** How many dimensions do the embeddings have?

In [ ]:
vect_articles.shape

(500, 7978)

Now, let's use the embeddings to look for similar articles to a search query.

Apply the vectorizer you fit earlier to this query string to get an embedding.

**Hint:** You can't pass a string to a vectorizer, but you can pass a list containing a string.

In [ ]:
query = "vector databases for retrieval augmented generation"

vect_query = vect.transform([query])

Now, we need to find the similarity between our query embedding and each vectorized article.

For this, you can use the [cosine similarity function from scikit-learn.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

Calculate the similarity between the query embedding and each article embedding and save the result to a variable named `similarity_scores`.

In [ ]:
similarity_scores = cosine_similarity(vect_query, vect_articles)

Now, we need to find the most similar results. To help with this, we can use the [argsort function from numpy](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html), which will give the indices sorted by value.

Use the argsort function to find the indices of the 5 most similar articles. Inspect their titles and abstracts. **Warning:** argsort sorts from smallest to largest.

In [ ]:
indices_sorted = np.argsort(similarity_scores)[0][-5:]
indices_sorted

# Alternate
# indices_sorted = np.argsort(-similarity_scores)[0,:5]

array([394, 486,  83, 289, 259])

In [ ]:
similarity_scores[0][indices_sorted]

array([0.15681251, 0.17057206, 0.18057878, 0.18926175, 0.26978155])

Try using a tfidf vectorizer. How do the results compare?

In [ ]:
vect = TfidfVectorizer().fit(articles['abstract'])
vect_articles = vect.transform(articles['abstract'])
query = "vector databases for retrieval augmented generation"
vect_query = vect.transform([query])
similarity_scores = cosine_similarity(vect_query, vect_articles)
indices_sorted = np.argsort(similarity_scores)[0][-5:]
similarity_scores[0][indices_sorted]

array([0.14238743, 0.15562194, 0.21030972, 0.26498602, 0.29961889])

### Method 2: Using a Pretrained Embedding Model

Now, let's compare how we do using the [all-MiniLM-L6-v2 embedding model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

This will create a 384-dimensional dense embedding of each sentence.

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = embedder.encode(sentences)
print(embeddings)

[[ 6.76569194e-02  6.34959713e-02  4.87130992e-02  7.93050006e-02
   3.74480896e-02  2.65282183e-03  3.93749885e-02 -7.09845545e-03
   5.93613982e-02  3.15369926e-02  6.00981042e-02 -5.29052094e-02
   4.06067520e-02 -2.59308629e-02  2.98427939e-02  1.12692360e-03
   7.35148415e-02 -5.03819063e-02 -1.22386619e-01  2.37028319e-02
   2.97265295e-02  4.24768366e-02  2.56337635e-02  1.99515908e-03
  -5.69190197e-02 -2.71597710e-02 -3.29035483e-02  6.60248995e-02
   1.19007207e-01 -4.58790772e-02 -7.26214647e-02 -3.25840227e-02
   5.23413233e-02  4.50553633e-02  8.25300440e-03  3.67024057e-02
  -1.39415190e-02  6.53918087e-02 -2.64272243e-02  2.06387354e-04
  -1.36643304e-02 -3.62810828e-02 -1.95043813e-02 -2.89738160e-02
   3.94270197e-02 -8.84090587e-02  2.62429030e-03  1.36713982e-02
   4.83062863e-02 -3.11566275e-02 -1.17329143e-01 -5.11690117e-02
  -8.85287672e-02 -2.18962654e-02  1.42986597e-02  4.44167443e-02
  -1.34815564e-02  7.43392110e-02  2.66382731e-02 -1.98762864e-02
   1.79191

Use this new embedder to vectorize the abstracts and then find the most similar to the query. How do the results compare to the other methods?

**Warning:** Creating embeddings for all of the articles may take a while.

In [ ]:
embeddings = embedder.encode(articles['abstract'].to_numpy())

# Alternate
# embeddings = embedder.encode(articles['abstract'].values)

In [ ]:
query = "vector databases for retrieval augmented generation"
query_embedding = embedder.encode([query])
similarity_scores = cosine_similarity(query_embedding, embeddings)
indices_sorted = np.argsort(similarity_scores)[0][-5:]
similarity_scores[0][indices_sorted]

array([0.41446638, 0.4343459 , 0.4514608 , 0.45547363, 0.47817037],
      dtype=float32)

### FAISS

The [Faiss library](https://faiss.ai/index.html) is a library for efficient similarity search and clustering of dense vectors. It can be used to automate the process of finding the most similar abstracts.

If we want to use cosine similarity, we need to use the Inner Product. We also need to normalize our vectors so that they all have length 1.

Use the [normalize function](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.normalize.html) to normalize both the abstract vectors and the query vector.

In [ ]:
# Not necessary due to prior step already normalizing
normalized_embeddings = normalize(embeddings)
normalized_query = normalize(query_embedding)

Now, create an [IndexFlatIP object](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes#summary-of-methods) that has dimensions equal to the dimensionality of your vectors. Then add your normalized abstract vectors.

Hint: You can mimic the example [here](https://github.com/facebookresearch/faiss/wiki/Getting-started#building-an-index-and-adding-the-vectors-to-it), but substitute in the IndexFlatIP class.

In [ ]:
index = faiss.IndexFlatIP(normalized_embeddings.shape[1]) # build the index
print(index.is_trained)
index.add(normalized_embeddings) # add vectors to the index
print(index.ntotal)

True
500


Finally, use the [search function](https://github.com/facebookresearch/faiss/wiki/Getting-started#searching) on your index object to find the 5 most similar articles.

In [ ]:
k = 5                          # we want to see 5 nearest neighbors
D, I = index.search(normalized_embeddings[:5], k) # sanity check
print(I)
print(D)
D, I = index.search(normalized_query, k)     # actual search
print(I[:5])                   # neighbors of the 5 first queries
print(I[-5:])                  # neighbors of the 5 last queries

[[  0   1 270 313 372]
 [  1   0 138 385 372]
 [  2 385 308 106 240]
 [  3 257  52  19  90]
 [  4 132  31 377  27]]
[[0.99999994 0.7133904  0.66111064 0.6530498  0.619205  ]
 [1.0000001  0.7133904  0.6755993  0.65596807 0.6516855 ]
 [1.0000001  0.67804754 0.64844745 0.6444368  0.64188176]
 [1.0000001  0.643654   0.58513725 0.5709048  0.5660356 ]
 [1.         0.654927   0.65049505 0.6379887  0.61652064]]
[[259 107 233 209 247]]
[[259 107 233 209 247]]


In [ ]:
D[0][:5]

array([0.47817037, 0.4554736 , 0.4514608 , 0.4343459 , 0.41446638],
      dtype=float32)